# Low-Cost Sensor vs Reference Monitor

**Goal:** Compare a PurpleAir low-cost sensor against the nearest AURN reference
monitor to assess data reliability and understand sensor performance.

**API keys required:** `PURPLEAIR_API_KEY`

**Aeolus features demonstrated:**
- Cross-source data retrieval with `download()` using dict format
- `find_sites()` for spatial discovery across different networks
- Standard schema enabling seamless cross-source joins
- `metrics.time_average()` for temporal alignment
- `viz.plot_timeseries()` for multi-source overlay
- QA/QC ratification flags across sources

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Check required API key
if not os.environ.get("PURPLEAIR_API_KEY"):
    raise EnvironmentError(
        "PURPLEAIR_API_KEY is required for this notebook.\n"
        "Get a free key at https://develop.purpleair.com/\n"
        "Then add it to your .env file: PURPLEAIR_API_KEY=your_key_here"
    )

In [ ]:
import aeolus
from aeolus import metrics, viz
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Find a PurpleAir Sensor

Search for PurpleAir sensors in a London bounding box. We'll pick one that's
near an AURN reference monitor for a meaningful comparison.

In [ ]:
# Find PurpleAir sensors in central London
pa_sites = aeolus.portals.find_sites(
    "PURPLEAIR",
    bbox=(-0.2, 51.48, 0.0, 51.55),  # (min_lon, min_lat, max_lon, max_lat)
)

print(f"Found {len(pa_sites)} PurpleAir sensors")
pa_sites[["site_code", "site_name", "latitude", "longitude"]].head()

In [ ]:
# Pick a sensor
pa_sensor = pa_sites.iloc[0]
pa_code = pa_sensor["site_code"]
pa_lat = pa_sensor["latitude"]
pa_lon = pa_sensor["longitude"]

print(f"Selected PurpleAir: {pa_sensor['site_name']} ({pa_code})")
print(f"Location: ({pa_lat:.4f}, {pa_lon:.4f})")

## 2. Find the Nearest AURN Reference Monitor

Use `find_sites()` with the PurpleAir sensor's coordinates to find
the closest AURN reference-grade monitor.

In [ ]:
# Find nearest AURN site to the PurpleAir sensor
aurn_sites = aeolus.find_sites(
    "AURN",
    near=(pa_lat, pa_lon),
    radius_km=10,
)

aurn_site = aurn_sites.iloc[0]
aurn_code = aurn_site["site_code"]

print(f"Nearest AURN: {aurn_site['site_name']} ({aurn_code})")
print(f"Distance from PurpleAir: {aurn_site['distance_km']:.1f} km")

## 3. Download Data from Both Sources

Download from both networks in a single call using the dict format.
Aeolus normalises both to the same 8-column schema, making the join trivial.

In [ ]:
# Download 1 month from both sources
start = datetime(2024, 6, 1)
end = datetime(2024, 6, 30)

combined = aeolus.download(
    {
        "AURN": [aurn_code],
        "PURPLEAIR": [pa_code],
    },
    start_date=start,
    end_date=end,
)

print(f"Total records: {len(combined):,}")
print(f"\nRecords by source:")
print(combined.groupby("source_network").size())

In [ ]:
# Check ratification flags — shows data quality metadata
print("Ratification flags by source:")
print(combined.groupby(["source_network", "ratification"]).size())

## 4. Align Timestamps

Both sources produce UTC-aware hourly data (Aeolus standardises this).
We use `time_average()` to ensure consistent hourly alignment, then
pivot for a paired comparison.

In [ ]:
# Filter to PM2.5 and compute hourly means (handles any sub-hourly data)
pm25 = combined[combined["measurand"] == "PM2.5"]

hourly = metrics.time_average(pm25, freq="h")

# Pivot: one column per source
paired = (
    hourly
    .pivot_table(index="date_time", columns="source_network", values="value")
    .dropna()
)

print(f"Paired hourly observations: {len(paired)}")
print(f"\nBasic statistics:")
paired.describe().round(1)

## 5. Time Series Overlay

Plot both sensors on the same axis to visually assess agreement.

In [ ]:
# Time series overlay
fig = viz.plot_timeseries(
    pm25,
    pollutants=["PM2.5"],
    title="PM\u2082.\u2085: PurpleAir vs AURN Reference",
)
plt.show()

## 6. Scatter Plot & Regression

A scatter plot with 1:1 line and regression statistics is the standard
method for evaluating sensor performance against a reference.

In [ ]:
from scipy import stats

# Identify columns (source network names)
cols = paired.columns.tolist()
ref_col = [c for c in cols if "AURN" in c.upper()][0]
lcs_col = [c for c in cols if c != ref_col][0]

x = paired[ref_col].values
y = paired[lcs_col].values

# Linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
r_squared = r_value ** 2

# RMSE and bias
rmse = np.sqrt(np.mean((y - x) ** 2))
bias = np.mean(y - x)

print(f"R\u00b2 = {r_squared:.3f}")
print(f"Slope = {slope:.2f}, Intercept = {intercept:.1f}")
print(f"RMSE = {rmse:.1f} \u00b5g/m\u00b3")
print(f"Bias = {bias:+.1f} \u00b5g/m\u00b3 ({'overestimates' if bias > 0 else 'underestimates'})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(x, y, alpha=0.3, s=10, color="#3498db")

# 1:1 line
lim = max(x.max(), y.max()) * 1.1
ax.plot([0, lim], [0, lim], "k--", linewidth=0.8, label="1:1")

# Regression line
x_fit = np.linspace(0, lim, 100)
ax.plot(x_fit, slope * x_fit + intercept, "r-", linewidth=1.5,
        label=f"y = {slope:.2f}x + {intercept:.1f} (R\u00b2={r_squared:.3f})")

ax.set_xlabel(f"AURN Reference PM\u2082.\u2085 (\u00b5g/m\u00b3)")
ax.set_ylabel(f"PurpleAir PM\u2082.\u2085 (\u00b5g/m\u00b3)")
ax.set_title("Sensor vs Reference: Hourly PM\u2082.\u2085")
ax.legend()
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 7. Daily Aggregation Comparison

Low-cost sensors often perform better at daily resolution because
random noise averages out. Compare daily means.

In [ ]:
# Daily means using time_average
daily = metrics.time_average(pm25, freq="D")

daily_paired = (
    daily[daily["measurand"] == "PM2.5"]
    .pivot_table(index="date_time", columns="source_network", values="value")
    .dropna()
)

# Daily regression
xd, yd = daily_paired[ref_col].values, daily_paired[lcs_col].values
slope_d, intercept_d, r_d, _, _ = stats.linregress(xd, yd)

print(f"Daily R\u00b2 = {r_d**2:.3f} (vs hourly R\u00b2 = {r_squared:.3f})")
print(f"Daily RMSE = {np.sqrt(np.mean((yd - xd)**2)):.1f} \u00b5g/m\u00b3 (vs hourly RMSE = {rmse:.1f})")

## Summary

This notebook demonstrated:

1. **Cross-source discovery** — finding sensors and reference monitors spatially
2. **Multi-source download** — dict format for downloading from different networks in one call
3. **Standard schema** — seamless join on `date_time` because Aeolus normalises all sources
4. **Composable aggregation** — `time_average()` at different frequencies for hourly and daily comparisons
5. **QA/QC flags** — ratification metadata distinguishes validated from unvalidated data

### Interpreting results
- **R² > 0.7**: Reasonable agreement for a low-cost sensor
- **Positive bias**: PurpleAir tends to overestimate PM₂.₅ (common in humid conditions)
- **Better daily performance**: Expected — random noise cancels over 24h averages

### Next steps
- Apply a correction factor (e.g., EPA's PurpleAir correction)
- Extend to a full year to capture seasonal variation
- Compare Sensor.Community data as a third source